# 15 — Length and formatting controls (Phi-2)

**Research question:** does Delta ID = ID(incorrect) - ID(correct) remain positive when attempts within the same problem are matched for length and simple formatting features?

This notebook uses existing representations and **the same common_clean cohort as FINAL notebook 13**. No model generation or extraction is performed. The tokenizer measures text features. `pre_mean` and `pre_last` are the primary views for this question; full-sequence views provide comparisons.

Three specified analyses:
1. `prefix_length`: match the number of generated tokens before the selected numeric span.
2. `prefix_length_format`: additionally match multiline and arithmetic-symbol indicators and the numeric-span count category (0, 1, 2+).
3. `full_length_format`: a separate exploratory control for length and formatting of the entire stored generated continuation. It uses text after the prefix and must not be interpreted as a control based only on information available before the answer.

Length and formatting features exclude the prompt. Representations retain their original definitions, including the prompt. The caliper is `abs(log1p(length_C)-log1p(length_I)) <= 0.20`: the ratio of length+1 is at most exp(0.20), approximately 1.22. This is not complete control of style or content.

Each control has its own reference with **the same problems and attempt counts per problem**, sampled from common_clean without length/format constraints. The adjusted-minus-reference contrast describes a selection-associated difference, not a causal effect. Controls may retain different problem sets and must not be treated as identical cohorts.

**Execution:** numbered code cells 1-5 create audits and plans. The default `RUN_ID=False` skips expensive estimation. For a trial, set RUN_ID=True with QUICK=True; for FINAL, use QUICK=False and RUN_ID=True. After a restart, run in order; completed checkpoints are loaded. `read_checkpoint` is defined before the execution cell. Notebook 13 does not need to be rerun. If multiple FINAL directories exist, set SOURCE_DIR to the intended previous run directory.

Sampling percentiles and LOPO ranges are not confidence intervals. Pair-swap permutations provide only a descriptive reference: close matching does not establish exchangeability. No p-values are computed. Poor balance or substantial problem loss limits interpretation to the retained overlap subset. Do not tune the caliper based on the observed ID result; changes constitute a new, explicitly labeled analysis.

**English edition:** explanations, comments, and display/error messages are translated. Analysis settings, data paths, and historical implementation identifiers are retained for checkpoint compatibility. Those identifiers refer to the original analysis version, not a byte hash of this translated file.


In [ ]:
# 1 — Settings and location of the previous FINAL run.
# If a package is missing: %pip install torch numpy pandas pyarrow matplotlib tqdm transformers
import json, hashlib, platform, re, gc, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/msc_thesis')
SOURCE_DIR=None  # or Path('/content/drive/MyDrive/msc_thesis/results/aligned_four_views_phi2_v1/final/YOUR_SIGNATURE')
if SOURCE_DIR is None:
    candidates=sorted((BASE/'results/aligned_four_views_phi2_v1/final').glob('*/manifest.json'))
    if len(candidates)!=1:
        raise ValueError('Set SOURCE_DIR to the previous FINAL directory. Candidates: '+str([str(p.parent) for p in candidates]))
    SOURCE_DIR=candidates[0].parent
SOURCE_DIR=Path(SOURCE_DIR)
QUICK=True
RUN_ID=False
CALIPER=0.20
CONTROLS=['prefix_length','prefix_length_format','full_length_format']
VIEWS=['finals','means','pre_last','pre_mean']
K_VALUES=[10,20]; LAYERS=list(range(32)); CAP=10
REPS=5 if QUICK else 120; PERMS=10 if QUICK else 80
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
ORIGINAL=BASE/'data/representations/phi2/phi2_repr_paired_v1.pt'
PREFIX=BASE/'data/representations/phi2/phi2_preanswer_paired_v1.pt'
FROZEN=BASE/'data/frozen/phi2_dataset_frozen_v1.parquet'
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
source_manifest=json.loads((SOURCE_DIR/'manifest.json').read_text())
assert source_manifest['config']['quick'] is False
assert source_manifest['config']['pooled_reps']==120
assert source_manifest['config']['permutation_reps']==80
# Verify that the local caches are exactly the inputs used in the previous audit.
for path in [ORIGINAL,PREFIX,FROZEN]:
    old=[v for k,v in source_manifest['inputs'].items() if Path(k).name==path.name]
    assert len(old)==1 and digest(path)==old[0], f'Input differs: {path}'
print('Source:',SOURCE_DIR,'| Device:',DEVICE)

IMPLEMENTATION_SHA='735add51b0a629d5eabdbbf9585712671b25a8a507b70bbaffca3530b4b8c623'


In [ ]:
# 2 — Frozen cohort and text features. No ID estimation in this cell.
def bool_column(s):
    v=s.astype(str).str.lower()
    if not v.isin(['true','false','0','1']).all(): raise ValueError('Invalid Boolean column')
    return v.isin(['true','1'])
membership=pd.read_csv(SOURCE_DIR/'cohort_membership.csv')
for col in ['correct','excluded']: membership[col]=bool_column(membership[col])
assert not membership.duplicated(['problem_id','attempt_id']).any()
meta=membership.loc[~membership.excluded].copy().reset_index(drop=True)
keys=list(zip(meta.problem_id.astype(int),meta.attempt_id.astype(int)))
expected=np.asarray(source_manifest['common_keys'])[source_manifest['clean_indices']]
assert np.array_equal(np.asarray(keys),expected), 'The cohort does not match the previous manifest'
frozen=pd.read_parquet(FROZEN).set_index(['problem_id','attempt_id'])
assert frozen.index.is_unique
# Pin the tokenizer to the previously recorded revision when available.
revision=source_manifest.get('tokenizer_commit')
tokenizer=AutoTokenizer.from_pretrained('microsoft/phi-2',use_fast=True,**({'revision':revision} if revision else {}))
assert tokenizer.is_fast
NUM_RE=re.compile(r'-?\d+\.?\d*')
def shape_features(text, tag):
    text=text.strip()
    return {tag+'multiline':int('\n' in text),
            tag+'arithmetic':int(bool(re.search(r'[=+*/×÷−]|(?<=\d)\s*-\s*(?=\d)',text))),
            tag+'numeric_bin':min(2,len(NUM_RE.findall(text)))}
features=[]
for idx,key in enumerate(tqdm(keys,desc='Length / format audit')):
    row=frozen.loc[key]; text=str(row.generated_text); prompt=str(row.prompt)
    assert bool_column(pd.Series([row.correct])).iloc[0]==bool(meta.loc[idx,'correct'])
    # Do not infer the prompt boundary unless the text starts with the exact prompt.
    if not text.startswith(prompt): raise ValueError(f'generated_text does not start with the prompt: {key}')
    enc=tokenizer(text,add_special_tokens=False,truncation=True,max_length=512,return_offsets_mapping=True)
    offsets=enc['offset_mapping']; cut=int(meta.loc[idx,'prefix_T'])
    assert len(enc['input_ids'])==int(meta.loc[idx,'T_full']) and 0<cut<len(offsets)
    matches=list(NUM_RE.finditer(text)); match=matches[-1]
    overlap=[j for j,(a,b) in enumerate(offsets) if b>match.start() and a<match.end() and a!=b]
    assert overlap and overlap[0]==cut and match.start()>=len(prompt), key
    generated=[j for j,(a,b) in enumerate(offsets) if b>len(prompt) and b>a]
    end=max(b for a,b in offsets)
    # Use only text corresponding to stored tokens, within the truncation boundary.
    prefix_text=text[len(prompt):max(len(prompt),offsets[cut][0])]
    full_text=text[len(prompt):end]
    features.append(dict(generated_prefix_tokens=sum(j<cut for j in generated),
        generated_full_tokens=len(generated),truncated=int(len(tokenizer(text,add_special_tokens=False)['input_ids'])>512),
        **shape_features(prefix_text,'prefix_'),**shape_features(full_text,'full_')))
meta=pd.concat([meta,pd.DataFrame(features)],axis=1)
FEATURES=list(features[0])
display(meta.groupby('correct')[FEATURES].mean().T)
print('Common clean:',len(meta),'attempts;',meta.problem_id.nunique(),'problems')


In [ ]:
# 3 — Self-contained numerical and matching functions.
"""Aligned ID comparison: numerical and sampling core (NumPy, float64)."""
import hashlib
import numpy as np


def normalize_rows(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Expected a finite [N,D] array')
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    if (norm <= 1e-12).any():
        raise ValueError('Zero or near-zero input norm')
    return x / norm


def identical_groups(x):
    """Exact value equality, treating +0 and -0 as equal; no rounding/tolerance."""
    x = np.asarray(x, dtype=np.float64).copy()
    x[x == 0] = 0.0
    buckets = {}
    for i, row in enumerate(x):
        digest = hashlib.sha256(row.tobytes()).hexdigest()
        buckets.setdefault(digest, []).append(i)
    result = []
    for ids in buckets.values():
        if len(ids) > 1:
            if not all(np.array_equal(x[ids[0]], x[j]) for j in ids[1:]):
                raise RuntimeError('Unexpected fingerprint collision')
            result.append(ids)
    return result


def make_plans(problem_ids, labels, cap, reps, seed, permute=False):
    """All indices refer to the same cohort. A plan is shared by all views and k."""
    problem_ids, labels = np.asarray(problem_ids), np.asarray(labels, bool)
    if len(problem_ids) != len(labels) or cap < 1 or reps < 1:
        raise ValueError('Invalid matching inputs')
    rng = np.random.default_rng(seed)
    pools = []
    for pid in np.unique(problem_ids):
        c = np.flatnonzero((problem_ids == pid) & labels)
        i = np.flatnonzero((problem_ids == pid) & ~labels)
        count = min(cap, len(c), len(i))
        if count == 0:
            raise ValueError(f'Problem {pid} lost a class; cannot match')
        pools.append((pid, c, i, count))
    plans = []
    for _ in range(reps):
        C, I = [], []
        for pid, c, i, count in pools:
            cc = rng.choice(c, count, replace=False)
            ii = rng.choice(i, count, replace=False)
            if permute:
                both = rng.permutation(np.concatenate([cc, ii]))
                cc, ii = both[:count], both[count:]
            C.extend(cc); I.extend(ii)
        C, I = np.asarray(C, int), np.asarray(I, int)
        assert len(np.unique(np.r_[C, I])) == len(C) + len(I)
        assert np.array_equal(problem_ids[C], problem_ids[I])
        if not permute:
            assert labels[C].all() and (~labels[I]).all()
        plans.append((C, I))
    return plans


def local_lb(neighbors, k):
    """Original mean-of-local-inverses formula; no epsilon clipping/jitter/drops."""
    a = np.asarray(neighbors, dtype=np.float64)[..., :k]
    if k < 2 or a.shape[-1] != k or not np.isfinite(a).all() or (a <= 1e-12).any():
        raise FloatingPointError('Invalid/zero kNN distance or insufficient neighbors')
    denominator = np.log(a[..., -1:] / a[..., :-1]).mean(axis=-1)
    if not np.isfinite(denominator).all() or (denominator <= 1e-12).any():
        raise FloatingPointError('Degenerate LB denominator')
    local = 1.0 / denominator
    if not np.isfinite(local).all():
        raise FloatingPointError('Nonfinite local ID')
    return local


def class_statistics(D, indices, cohort_pids, all_pids, k_values, with_lopo=True):
    """Exact LOPO via extra cached neighbors, validated against full recomputation.

    Removing a problem removes at most m points from a class. Every surviving
    query's new first k neighbors must be among its original first k+m neighbors.
    Distances to other observations do not change under row-wise L2 normalization.
    """
    indices = np.asarray(indices, int)
    group = np.asarray(cohort_pids)[indices]
    all_pids = np.asarray(all_pids)
    sub = np.asarray(D[np.ix_(indices, indices)], dtype=np.float64)
    np.fill_diagonal(sub, np.inf)
    n, max_k = len(indices), max(k_values)
    max_removed = max(int((group == p).sum()) for p in all_pids)
    if n <= max_k or (with_lopo and n - max_removed <= max_k):
        raise ValueError('Insufficient pooled/LOPO support for requested k')
    needed = min(n - 1, max_k + (max_removed if with_lopo else 0))
    nn = np.argpartition(sub, needed - 1, axis=1)[:, :needed]
    distances = np.take_along_axis(sub, nn, axis=1)
    order = np.argsort(distances, axis=1, kind='stable')
    nn = np.take_along_axis(nn, order, axis=1)
    distances = np.take_along_axis(distances, order, axis=1)
    observed, diag = [], []
    for k in k_values:
        local = local_lb(distances, k)
        observed.append(local.mean())
        diag.append([local.min(), np.median(local), local.max()])
    lopo = np.empty((len(all_pids), len(k_values))) if with_lopo else None
    if with_lopo:
        neighbor_groups = group[nn]
        for j, pid in enumerate(all_pids):
            keep_queries = group != pid
            updated = np.where(neighbor_groups[keep_queries] == pid,
                               np.inf, distances[keep_queries])
            updated.sort(axis=1)
            for ki, k in enumerate(k_values):
                lopo[j, ki] = local_lb(updated, k).mean()
    return np.asarray(observed), lopo, np.asarray(diag)


def evaluate_layer(D, pids, pooled_plans, permutation_plans, k_values, progress=None):
    all_pids = np.unique(np.asarray(pids)[np.asarray(pooled_plans[0]).ravel()])
    nR, nB, nK, nP = len(pooled_plans), len(permutation_plans), len(k_values), len(all_pids)
    observed = np.empty((nR, 2, nK))
    lopo = np.empty((nR, nP, 2, nK))
    diagnostics = np.empty((nR, 2, nK, 3))
    iterator = enumerate(pooled_plans)
    if progress is not None:
        iterator = progress(iterator, total=nR, desc='Matched + LOPO', leave=False)
    for r, plan in iterator:
        for side, indices in enumerate(plan):
            try:
                obs, leave, diag = class_statistics(D, indices, pids, all_pids, k_values)
            except (ValueError, FloatingPointError) as exc:
                raise RuntimeError(f'pooled draw={r}, class={side}: {exc}') from exc
            observed[r, side] = obs
            lopo[r, :, side] = leave
            diagnostics[r, side] = diag
    permutation = np.empty((nB, 2, nK))
    iterator = enumerate(permutation_plans)
    if progress is not None:
        iterator = progress(iterator, total=nB, desc='Permutation', leave=False)
    for b, plan in iterator:
        for side, indices in enumerate(plan):
            try:
                permutation[b, side] = class_statistics(
                    D, indices, pids, all_pids, k_values, with_lopo=False)[0]
            except (ValueError, FloatingPointError) as exc:
                raise RuntimeError(f'permutation draw={b}, class={side}: {exc}') from exc
    return dict(observed=observed, lopo=lopo, permutation=permutation,
                diagnostics=diagnostics, problem_ids=all_pids)


def summarize_draws(x):
    x = np.asarray(x, np.float64)
    if not np.isfinite(x).all():
        raise FloatingPointError('Invalid values cannot be silently omitted')
    return x.mean(axis=0), np.percentile(x, 2.5, axis=0), np.percentile(x, 97.5, axis=0)

import numpy as np

def maximum_pairs(valid, rng):
    """Randomized augmenting paths: maximum cardinality, not uniform sampling."""
    valid = np.asarray(valid, bool)
    mate = np.full(valid.shape[1], -1, int)
    adjacency = [rng.permutation(np.flatnonzero(row)) for row in valid]
    def visit(a, seen):
        for b in adjacency[a]:
            if seen[b]: continue
            seen[b] = True
            if mate[b] < 0 or visit(mate[b], seen):
                mate[b] = a
                return True
        return False
    for a in rng.permutation(valid.shape[0]):
        visit(a, np.zeros(valid.shape[1], bool))
    return np.asarray([(a,b) for b,a in enumerate(mate) if a >= 0], int).reshape(-1,2)

def build_control(meta, control, reps, seed, cap=10, caliper=0.2):
    # Lengths exclude the prompt. log1p permits a zero-token generated prefix.
    full = control == 'full_length_format'
    length = 'generated_full_tokens' if full else 'generated_prefix_tokens'
    style = ('full_' if full else 'prefix_')
    pools, report = [], []
    for pid,g in meta.groupby('problem_id',sort=True):
        c = g.index[g.correct].to_numpy(); i = g.index[~g.correct].to_numpy()
        gap = np.abs(np.log1p(meta.loc[c,length].to_numpy())[:,None] - np.log1p(meta.loc[i,length].to_numpy())[None,:])
        valid = gap <= caliper
        if control != 'prefix_length':
            for col in [style+'multiline',style+'arithmetic',style+'numeric_bin']:
                valid &= meta.loc[c,col].to_numpy()[:,None] == meta.loc[i,col].to_numpy()[None,:]
        n = min(cap,len(maximum_pairs(valid,np.random.default_rng(seed)) ))
        report.append(dict(control=control,problem_id=int(pid),correct_available=len(c),incorrect_available=len(i),matched_pairs=n))
        if n: pools.append((c,i,valid,n))
    rng = np.random.default_rng(seed)
    adjusted, reference = [], []
    for _ in range(reps):
        ac,ai,bc,bi = [],[],[],[]
        for c,i,valid,n in pools:
            pairs = maximum_pairs(valid,rng)
            pairs = pairs[rng.choice(len(pairs),n,replace=False)]
            ac.extend(c[pairs[:,0]]); ai.extend(i[pairs[:,1]])
            bc.extend(rng.choice(c,n,replace=False)); bi.extend(rng.choice(i,n,replace=False))
        adjusted.append([ac,ai]); reference.append([bc,bi])
    return np.asarray(adjusted,int), np.asarray(reference,int), report

def pair_swap(plans, reps, seed):
    rng=np.random.default_rng(seed); result=[]
    for b in range(reps):
        c,i=plans[b % len(plans)]
        swap=rng.integers(0,2,len(c)).astype(bool)
        result.append([np.where(swap,i,c),np.where(swap,c,i)])
    return np.asarray(result,int)


In [ ]:
# 4 — Plans and retention checks, before inspecting ID results.
PLANS={}; retention=[]; eligibility=[]
for ci,control in enumerate(CONTROLS):
    adjusted,reference,rows=build_control(meta,control,REPS,3100+ci,cap=CAP,caliper=CALIPER)
    retention.extend(rows)
    retained=[r for r in rows if r['matched_pairs']>0]
    n=sum(r['matched_pairs'] for r in retained)
    # LOPO must retain > max(k) points. This is a feasibility minimum, not a reliability guarantee.
    feasible=len(retained)>=2 and n-max([r['matched_pairs'] for r in retained],default=0)>max(K_VALUES)
    eligibility.append(dict(control=control,retained_problems=len(retained),total_problems=meta.problem_id.nunique(),
                            points_per_class=n,feasible=feasible))
    if not feasible: continue
    for arm,plans in [('adjusted',adjusted),('reference',reference)]:
        name=control+'__'+arm
        for c,i in plans:
            assert len(np.unique(np.r_[c,i]))==len(c)+len(i)
            assert meta.loc[c,'correct'].all() and (~meta.loc[i,'correct']).all()
            assert np.array_equal(meta.loc[c,'problem_id'],meta.loc[i,'problem_id'])
        PLANS[name]=dict(pooled=plans,permutation=pair_swap(plans,PERMS,4100+ci))
retention=pd.DataFrame(retention); eligibility=pd.DataFrame(eligibility)
display(eligibility)
display(retention[retention.matched_pairs==0])
# Count distinct plans to identify repetitions that add no selection variability.
for name,p in PLANS.items():
    unique=len({tuple(tuple(sorted(side)) for side in draw) for draw in p['pooled']})
    print(name,'unique class selections:',unique,'/',REPS)


In [ ]:
# 5 — Balance diagnostics and manifest/plan export. Stop here for the initial review.
# Standardization uses the full common_clean cohort SD, with the same denominator in both arms.
# For constant features, standardized differences are undefined; save NaN and the raw gap explicitly.
balance=[]
for name,p in PLANS.items():
    for feature in FEATURES:
        x=meta[feature].to_numpy(float); sd=x.std(ddof=1)
        gaps=np.asarray([x[c].mean()-x[i].mean() for c,i in p['pooled']])
        balance.append(dict(analysis=name,feature=feature,correct_minus_incorrect=gaps.mean(),
            mean_abs_gap=np.abs(gaps).mean(),mean_abs_standardized_gap=np.abs(gaps).mean()/sd if sd>0 else np.nan,
            constant_feature=sd==0))
balance=pd.DataFrame(balance)
config=dict(version='length_shape_v1',quick=QUICK,reps=REPS,perms=PERMS,caliper=CALIPER,cap=CAP,
            controls=CONTROLS,views=VIEWS,k_values=K_VALUES,layers=LAYERS,normalize=True,
            matching_seeds=[3100+i for i in range(len(CONTROLS))],permutation_seeds=[4100+i for i in range(len(CONTROLS))],
            implementation_sha256=IMPLEMENTATION_SHA)
manifest=dict(config=config,source_manifest_sha256=digest(SOURCE_DIR/'manifest.json'),
    cohort_sha256=digest(SOURCE_DIR/'cohort_membership.csv'),tokenizer_revision=revision,
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,torch=torch.__version__),
    device=DEVICE,features_sha256=hashlib.sha256(meta.to_csv(index=False).encode()).hexdigest(),
    plans_sha256={name:hashlib.sha256(p['pooled'].tobytes()+p['permutation'].tobytes()).hexdigest() for name,p in PLANS.items()})
SIGNATURE=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()
RES_DIR=BASE/'results/length_shape_phi2_v1'/('quick' if QUICK else 'final')/SIGNATURE[:12]
RES_DIR.mkdir(parents=True,exist_ok=True)
(RES_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))
meta.to_csv(RES_DIR/'features.csv',index=False); retention.to_csv(RES_DIR/'retention.csv',index=False)
eligibility.to_csv(RES_DIR/'eligibility.csv',index=False); balance.to_csv(RES_DIR/'balance.csv',index=False)
for name,p in PLANS.items(): np.savez_compressed(RES_DIR/(name+'_plans.npz'),cohort_keys=np.asarray(keys),**p)
if len(balance):
    display(balance.pivot(index='feature',columns='analysis',values='mean_abs_standardized_gap').round(3))
    fig,ax=plt.subplots(figsize=(12,5))
    balance.pivot(index='feature',columns='analysis',values='mean_abs_standardized_gap').plot.bar(ax=ax)
    ax.set_ylabel('Mean |class mean gap| / common-cohort SD'); fig.tight_layout()
    fig.savefig(RES_DIR/'balance.png',dpi=150); plt.show()
print('Results:',RES_DIR)
print('ID estimation enabled:',RUN_ID,'| QUICK:',QUICK)


In [ ]:
# 6 — Checkpoint functions are always available, independently of RUN_ID.
(RES_DIR/'checkpoints').mkdir(exist_ok=True)
def checkpoint_path(name,view,layer):
    return RES_DIR/'checkpoints'/f'{name}__{view}__L{layer:02d}.npz'
def read_checkpoint(path):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['signature'].item())==SIGNATURE, 'Checkpoint signature mismatch'
        result={k:z[k].copy() for k in z.files if k!='signature'}
    assert result['observed'].shape==(REPS,2,len(K_VALUES))
    assert result['lopo'].shape==(REPS,len(result['problem_ids']),2,len(K_VALUES))
    assert result['permutation'].shape==(PERMS,2,len(K_VALUES))
    for field in ['observed','lopo','permutation','diagnostics']:
        assert np.isfinite(result[field]).all(), f'Invalid checkpoint: {path} {field}'
    return result
@torch.no_grad()
def make_distance(rows,view,layer):
    raw=np.stack([r[view][layer].double().numpy() for r in rows])
    x=torch.from_numpy(normalize_rows(raw)).to(DEVICE)
    d=np.empty((len(x),len(x)),dtype=np.float64)
    for start in range(0,len(x),256):
        d[start:start+256]=torch.cdist(x[start:start+256],x,p=2,compute_mode='donot_use_mm_for_euclid_dist').cpu().numpy()
    np.fill_diagonal(d,np.inf)
    off=~np.eye(len(d),dtype=bool)
    if (d[off]<=1e-12).any() or not np.isfinite(d[off]).all():
        raise FloatingPointError('Invalid distance. Additional attempts are not removed automatically.')
    return d


In [ ]:
# 7 — Compute the new controls. Skip completed checkpoints.
if RUN_ID:
    if not PLANS: raise ValueError('No control has sufficient overlap; review the audit.')
    for view in VIEWS:
        pending_layers=[ell for ell in LAYERS if any(not checkpoint_path(n,view,ell).exists() for n in PLANS)]
        if not pending_layers: continue
        path=ORIGINAL if view in ['finals','means'] else PREFIX
        cache=torch.load(path,map_location='cpu',weights_only=False)
        lookup={(int(r['problem_id']),int(r['attempt_id'])):r for r in cache}
        assert len(lookup)==len(cache)
        records=[lookup[key] for key in keys]
        for j,r in enumerate(records):
            assert bool_column(pd.Series([r['correct']])).iloc[0]==bool(meta.loc[j,'correct'])
            assert tuple(r[view].shape)==(32,2560)
        del cache,lookup
        for ell in tqdm(pending_layers,desc=view):
            D=make_distance(records,view,ell)
            for name,p in PLANS.items():
                target=checkpoint_path(name,view,ell)
                if target.exists(): read_checkpoint(target); continue
                result=evaluate_layer(D,meta.problem_id.to_numpy(),p['pooled'],p['permutation'],K_VALUES,progress=tqdm)
                temp=target.with_suffix('.tmp')
                with temp.open('wb') as f: np.savez_compressed(f,signature=np.asarray(SIGNATURE),**result)
                temp.replace(target)
            del D; gc.collect()
        del records; gc.collect()
else:
    print('Audit complete. ID estimation was not run because RUN_ID=False.')


In [ ]:
# 8 — Load saved results without repeating the computation in cell 7.
RESULTS={}; missing=[]; rows=[]; late=[]
for name in PLANS:
    for view in VIEWS:
        paths=[checkpoint_path(name,view,ell) for ell in LAYERS]
        if not all(p.exists() for p in paths): missing.append((name,view)); continue
        blocks=[read_checkpoint(p) for p in paths]
        assert all(np.array_equal(b['problem_ids'],blocks[0]['problem_ids']) for b in blocks)
        observed=np.stack([b['observed'] for b in blocks],axis=-1)
        lo=np.stack([b['lopo'] for b in blocks],axis=-1)
        perm=np.stack([b['permutation'] for b in blocks],axis=-1)
        delta=observed[:,1]-observed[:,0]; lopo=(lo[:,:,1]-lo[:,:,0]).mean(axis=0)
        null=perm[:,1]-perm[:,0]
        RESULTS[name,view]=dict(delta=delta,lopo=lopo)
        for ki,k in enumerate(K_VALUES):
            for li,ell in enumerate(LAYERS):
                d=delta[:,ki,li]; j=lopo[:,ki,li]; q=null[:,ki,li]
                rows.append(dict(analysis=name,representation=view,k=k,layer=ell,delta_id=d.mean(),
                    sampling_p025=np.percentile(d,2.5),sampling_p975=np.percentile(d,97.5),
                    lopo_min=j.min(),lopo_max=j.max(),perm_mean=q.mean(),
                    perm_p025=np.percentile(q,2.5),perm_p975=np.percentile(q,97.5)))
            d=delta[:,ki,24:32].mean(axis=-1); j=lopo[:,ki,24:32].mean(axis=-1)
            late.append(dict(analysis=name,representation=view,k=k,window='L24-L31',delta_id=d.mean(),
                sampling_p025=np.percentile(d,2.5),sampling_p975=np.percentile(d,97.5),lopo_min=j.min(),lopo_max=j.max()))
summary=pd.DataFrame(rows); late_summary=pd.DataFrame(late)
if rows:
    summary.to_csv(RES_DIR/'id_summary.csv',index=False)
    late_summary.to_csv(RES_DIR/'late_summary.csv',index=False)
    display(late_summary.round(3))
print('Incomplete combinations:',missing)


In [ ]:
# 9 — Adjusted-minus-reference contrasts and plots. Compare the same problems and sample sizes.
contrasts=[]
for control in CONTROLS:
    for view in VIEWS:
        a=RESULTS.get((control+'__adjusted',view)); b=RESULTS.get((control+'__reference',view))
        if a is None or b is None: continue
        for ki,k in enumerate(K_VALUES):
            d=(a['delta'][:,ki,24:32]-b['delta'][:,ki,24:32]).mean(axis=-1)
            j=(a['lopo'][:,ki,24:32]-b['lopo'][:,ki,24:32]).mean(axis=-1)
            contrasts.append(dict(control=control,representation=view,k=k,window='L24-L31',
                adjusted_minus_reference=d.mean(),sampling_p025=np.percentile(d,2.5),sampling_p975=np.percentile(d,97.5),
                lopo_min=j.min(),lopo_max=j.max()))
    for k in K_VALUES:
        if summary.empty: continue
        fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True)
        for ax,view in zip(axes.ravel(),VIEWS):
            for arm in ['reference','adjusted']:
                s=summary[(summary.analysis==control+'__'+arm)&(summary.representation==view)&(summary.k==k)]
                if s.empty: continue
                ax.plot(s.layer,s.delta_id,label=arm)
                ax.fill_between(s.layer,s.sampling_p025,s.sampling_p975,alpha=.15)
            ax.axhline(0,color='black',lw=.7); ax.set_title(view); ax.set_xlabel('Layer'); ax.set_ylabel('Delta ID')
            if ax.lines: ax.legend()
        fig.suptitle(f'{control}, k={k} — sampling percentiles, not CI')
        fig.tight_layout(); fig.savefig(RES_DIR/f'{control}_k{k}.png',dpi=150); plt.show()
if contrasts:
    contrasts=pd.DataFrame(contrasts); contrasts.to_csv(RES_DIR/'late_contrasts.csv',index=False)
    display(contrasts.round(3))
# Compact audit/results bundle, excluding raw representations and checkpoints.
with zipfile.ZipFile(RES_DIR/'review_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RES_DIR.iterdir()):
        if path.suffix in ['.csv','.json','.png']: z.write(path,path.name)
print('Review bundle:',RES_DIR/'review_bundle.zip')


## Reading the results

First inspect `eligibility.csv`, `retention.csv`, and `balance.csv`: how many problems are lost, how many points remain, and whether length/format differences decrease. Feasibility only means kNN/LOPO can be computed; small samples may still yield unstable estimates.

Compare each adjusted arm against its own reference. Positive adjusted Delta ID with acceptable balance indicates persistence under this particular control. Attenuation suggests an association with the selected properties but does not establish causation. A sign change accompanied by substantial overlap loss does not establish that the original result was wrong. LOPO covers only retained problems, without rematching after omission.

Repeated matching runs are neither independent new observations nor uniform samples of all possible matchings. Repeated plans may be unavoidable when only one valid pair exists. Only simple formatting features are controlled; lexical content and earlier answer mentions remain possible explanations. Conclusions are descriptive and conditional on selection, not evidence that the model knows its correctness label.
